In [2]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch
from matplotlib import font_manager
import matplotlib as mpl
from pathlib import Path
import os

# 设置中文字体
if os.name == 'nt':
    plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
elif os.name == 'posix': # macos 或 linux
    plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Arial Unicode MS', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False

# # Chinese font
# font_candidates = [
#     "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
#     "/usr/share/fonts/opentype/noto/NotoSerifCJK-Regular.ttc",
# ]
# font_path = next((p for p in font_candidates if os.path.exists(p)), None)
# if font_path:
#     font_manager.fontManager.addfont(font_path)
#     mpl.rcParams["font.family"] = font_manager.FontProperties(fname=font_path).get_name()
# mpl.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(15, 9))
ax.set_xlim(0, 15)
ax.set_ylim(0, 9)
ax.axis("off")

def box(x, y, w, h, text, fontsize=12, lw=1.6, fc="white"):
    r = Rectangle((x, y), w, h, linewidth=lw, edgecolor="black", facecolor=fc)
    ax.add_patch(r)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center",
            fontsize=fontsize, linespacing=1.25)
    return r

def arrow(x1, y1, x2, y2, text=None, fontsize=10.5, rad=0):
    a = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="->",
        mutation_scale=15,
        linewidth=1.45,
        color="black",
        connectionstyle=f"arc3,rad={rad}"
    )
    ax.add_patch(a)
    if text:
        ax.text(
            (x1+x2)/2, (y1+y2)/2 + 0.16,
            text,
            ha="center", va="center", fontsize=fontsize,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.95)
        )
    return a

def line(x1, y1, x2, y2, lw=1.4):
    ax.plot([x1, x2], [y1, y2], color="black", linewidth=lw)

# Title
ax.text(7.5, 8.55, "EEG 8 通道：CHx 相对于 SRB2，SRB2 相对于 BIAS 的参考关系图",
        ha="center", va="center", fontsize=19, fontweight="bold")

# Body/head model
body = Circle((3.1, 4.7), 2.05, fill=False, linewidth=2.0)
ax.add_patch(body)
ax.text(3.1, 7.05, "人体 / 头皮电极系统", ha="center", fontsize=14, fontweight="bold")

# Electrode positions
signal_positions = [
    ("CH1", 1.85, 5.75), ("CH2", 2.65, 6.15), ("CH3", 3.55, 6.15), ("CH4", 4.35, 5.75),
    ("CH5", 1.85, 4.65), ("CH6", 2.65, 4.25), ("CH7", 3.55, 4.25), ("CH8", 4.35, 4.65),
]
for label, x, y in signal_positions:
    ax.add_patch(Circle((x, y), 0.13, color="black"))
    ax.text(x, y+0.22, label, ha="center", va="bottom", fontsize=10)

# SRB2 reference and BIAS
srb2_xy = (3.1, 3.45)
bias_xy = (3.1, 2.55)
ax.add_patch(Circle(srb2_xy, 0.16, color="black"))
ax.text(srb2_xy[0]-0.05, srb2_xy[1]-0.38, "SRB2\n参考电极 REF", ha="center", va="top", fontsize=11)
ax.add_patch(Circle(bias_xy, 0.16, color="black"))
ax.text(bias_xy[0], bias_xy[1]-0.38, "BIAS\n共模驱动电极", ha="center", va="top", fontsize=11)

# Indicate all channels refer to SRB2
for label, x, y in signal_positions:
    arrow(x, y-0.05, srb2_xy[0], srb2_xy[1]+0.1, rad=0.05 if x < 3.1 else -0.05)

ax.text(3.1, 1.85, "EEG 有效量：V_CHx − V_SRB2", ha="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black"))

# ADS1299 block
# box(7.2, 1.3, 6.5, 6.5, "ADS1299 / EEG-8ch 模拟前端", fontsize=16, lw=2)

# Inputs table style
box(7.65, 6.85, 1.45, 0.45, "CH1P", fontsize=10)
box(7.65, 6.35, 1.45, 0.45, "CH2P", fontsize=10)
box(7.65, 5.85, 1.45, 0.45, "CH3P", fontsize=10)
box(7.65, 5.35, 1.45, 0.45, "CH4P", fontsize=10)
box(7.65, 4.85, 1.45, 0.45, "CH5P", fontsize=10)
box(7.65, 4.35, 1.45, 0.45, "CH6P", fontsize=10)
box(7.65, 3.85, 1.45, 0.45, "CH7P", fontsize=10)
box(7.65, 3.35, 1.45, 0.45, "CH8P", fontsize=10)

# SRB2 and internal mux to negative inputs
box(9.65, 4.05, 1.45, 1.1, "SRB2\n公共参考总线", fontsize=11)
for yy in [7.075, 6.575, 6.075, 5.575, 5.075, 4.575, 4.075, 3.575]:
    line(9.1, yy, 9.65, 4.6, lw=1.0)
box(11.7, 4.65, 1.45, 1.15, "8 路 PGA / ADC\n测量差分", fontsize=11)
arrow(11.1, 4.6, 11.7, 5.2, "CHxP − SRB2", fontsize=10)

# Bias block
box(9.65, 2.0, 1.45, 0.85, "BIASOUT", fontsize=11)
box(11.7, 2.0, 1.45, 0.85, "BIAS 放大器\n共模反馈", fontsize=10)
arrow(11.7, 2.43, 11.1, 2.43, "输出弱反馈", fontsize=10)
arrow(11.7, 4.65, 12.42, 2.85, "提取/反馈共模", fontsize=10, rad=-0.15)

# Wires body to ADS
# channel bundle
arrow(4.8, 5.3, 7.65, 5.35, "8 个信号电极 → CH1P~CH8P", fontsize=11)
arrow(srb2_xy[0]+0.15, srb2_xy[1], 9.65, 4.6, "SRB2 作为公共参考输入", fontsize=11)
arrow(9.65, 2.43, bias_xy[0]+0.12, bias_xy[1], "BIAS 驱动人体整体共模", fontsize=11, rad=-0.08)
# Reference hierarchy annotations
box(0.7, 0.45, 4.15, 0.9, "第一层：EEG 通道参考\nCH1~CH8 都是相对于 SRB2：V_EEGx = V_CHx − V_SRB2", fontsize=10.5)
box(5.35, 0.45, 4.15, 0.9, "第二层：人体共模参考\nSRB2 自己不是绝对零点，它随人体整体电位漂浮", fontsize=10.5)
box(10.0, 0.45, 4.15, 0.9, "第三层：BIAS 稳定共模\nBIAS 不是测量参考，而是把人体共模拉回 ADS1299 工作范围", fontsize=10.5)
# Important note
ax.text(7.5, 8.1, "重点：CHx 参考 SRB2；SRB2 的绝对电位由人体-电路共模关系决定；BIAS 用反馈稳定这个共模关系",
        ha="center", fontsize=12.5)
# Ground rail
ax.plot([8.4, 12.8], [1.55, 1.55], color="black", linewidth=1.6)
ax.text(10.6, 1.28, "ADS1299 模拟地 / 供电工作范围", ha="center", fontsize=10.5)
plt.tight_layout()
out = Path("../outputs/eeg_8ch_srb2_bias_reference_relationship.png")
fig.savefig(out, dpi=220, bbox_inches="tight")
plt.close(fig)
out.as_posix()


'../outputs/eeg_8ch_srb2_bias_reference_relationship.png'

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, Circle
from matplotlib import font_manager
import matplotlib as mpl
from pathlib import Path
import subprocess, os, glob


fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

def box(x, y, w, h, text, fontsize=13, lw=1.8, fc='none'):
    r = Rectangle((x, y), w, h, linewidth=lw, edgecolor='black', facecolor=fc)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', fontsize=fontsize, linespacing=1.35)
    return r

def arrow(x1, y1, x2, y2, text=None, fontsize=11, rad=0.0):
    a = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='->', mutation_scale=16,
                        linewidth=1.6, connectionstyle=f"arc3,rad={rad}", color='black')
    ax.add_patch(a)
    if text:
        ax.text((x1+x2)/2, (y1+y2)/2 + 0.2, text, ha='center', va='center', fontsize=fontsize,
                bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='none', alpha=0.9))
    return a

ax.text(7, 7.55, "ADS1299 EEG 系统中的三层参考关系图", ha='center', va='center',
        fontsize=20, fontweight='bold')

body = Circle((3.1, 4.2), 1.55, fill=False, linewidth=2)
ax.add_patch(body)
ax.text(3.1, 5.95, "人体 / 头皮电极系统", ha='center', fontsize=14, fontweight='bold')

electrodes = {
    "Fp1\n信号电极": (2.35, 4.75),
    "C3\n信号电极": (3.75, 4.75),
    "REF\n参考电极": (3.05, 3.95),
    "BIAS\n驱动电极": (3.05, 3.25),
}
for label, (x, y) in electrodes.items():
    ax.add_patch(Circle((x, y), 0.18, color='black'))
    ax.text(x, y-0.42, label, ha='center', va='top', fontsize=11)

box(7.1, 2.0, 5.4, 4.6, "ADS1299 模拟前端", fontsize=16, lw=2)
box(7.55, 5.25, 1.6, 0.65, "CH1P", fontsize=12)
box(7.55, 4.35, 1.6, 0.65, "CH2P", fontsize=12)
box(7.55, 3.45, 1.6, 0.65, "SRB1 / REF\n公共负输入", fontsize=11)
box(7.55, 2.55, 1.6, 0.65, "BIASOUT", fontsize=12)
box(10.0, 4.25, 1.7, 1.15, "PGA / ΔΣ ADC\n测差分", fontsize=12)
box(10.0, 2.55, 1.7, 1.05, "BIAS 放大器\n共模反馈", fontsize=12)
box(10.0, 5.75, 1.7, 0.55, "VREFP / VREFN\nADC转换参考", fontsize=10)

ax.plot([8.1, 11.8], [1.35, 1.35], color='black', linewidth=1.8)
ax.text(9.95, 1.05, "ADS1299 模拟地 / AVSS-AVDD 工作范围", ha='center', fontsize=12)

arrow(2.35, 4.75, 7.55, 5.58, "Fp1 相对电路地有输入电压")
arrow(3.75, 4.75, 7.55, 4.68, "C3 相对电路地有输入电压")
arrow(3.05, 3.95, 7.55, 3.78, "REF 送入公共参考输入")
arrow(7.55, 2.88, 3.05, 3.25, "BIAS 弱反馈驱动人体共模", rad=-0.1)
arrow(9.15, 5.58, 10.0, 4.95)
arrow(9.15, 4.68, 10.0, 4.95)
arrow(9.15, 3.78, 10.0, 4.65)
arrow(10.85, 3.6, 10.85, 4.25, "稳定共模后再测差分", fontsize=10)
arrow(10.85, 5.75, 10.85, 5.4, "决定数字码换算", fontsize=10)

box(0.8, 0.35, 3.7, 1.05, "第 1 层：电极输入电压\n每个电极都相对于 ADS1299 模拟地表现为一个电压", fontsize=10.5)
box(5.15, 0.35, 3.7, 1.05, "第 2 层：EEG 测量参考\n通道结果 = 信号电极 − REF 电极", fontsize=10.5)
box(9.5, 0.35, 3.7, 1.05, "第 3 层：ADC 转换参考\n差分电压再相对于 VREFP/VREFN 转成数字码", fontsize=10.5)

ax.text(5.2, 6.55, "输入共模：V_CM = (V_CH + V_REF) / 2", fontsize=12,
        bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='black'))
ax.text(5.2, 6.0, "有效 EEG：V_EEG = V_CH − V_REF", fontsize=12,
        bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='black'))
ax.text(7, 7.05, "REF 定义“怎么相减”；BIAS 稳定“整体漂到哪里”；VREF 定义“ADC 怎么把差分电压量化”",
        ha='center', fontsize=13)

plt.tight_layout()
out = Path('../outputs/ads1299_eeg_reference_relationship.png')
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.close(fig)
out.as_posix()